<a href="https://colab.research.google.com/github/AKDGrant/weather-agent/blob/main/Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Weather Agent
Fetches live weather data and summarizes it using a free Hugging Face model. Demo shows multiple cities.

In [ ]:
!pip install langchain_community

In [ ]:
!pip install transformers accelerate langchain

In [3]:
# ======== Imports ========
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import requests
from dotenv import load_dotenv
import os

In [4]:
def get_weather(city: str):
    url = f"https://wttr.in/{city}?format=j1"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        response = requests.get(url, headers=headers).json()
        current = response['current_condition'][0]
        condition = current['weatherDesc'][0]['value']
        temp_C = current['temp_C']
        return f"{city}: {condition}, {temp_C}°C"
    except:
        return f"{city}: Weather data unavailable"

In [ ]:
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

def summarize(text):
    prompt = f"Summarize this weather report in a readable sentence: {text}"
    result = pipe(prompt, do_sample=False)
    return result[0]['generated_text']

In [6]:

def weather_report(cities):
    for city in cities:
        raw = get_weather(city)
        summary = summarize(raw)
        print(f"--- {city} ---")
        print("Raw weather:", raw)
        print("Summarized:", summary)
        print("\n")

In [7]:
# ======== Demo: Multiple Cities ========
cities = ["New York,US", "London,GB", "Tokyo,JP"]
weather_report(cities)

--- New York,US ---
Raw weather: New York,US: Sunny, 23°C
Summarized: New York, US: Sunny, 23°C


--- London,GB ---
Raw weather: London,GB: Partly cloudy, 13°C
Summarized: London, UK: Partly cloudy, 13°C


--- Tokyo,JP ---
Raw weather: Tokyo,JP: Partly cloudy, 25°C
Summarized: Tokyo: Partly cloudy, 25°C


